# **Model1 : RandomForest/ Lgbm/ GradientBoosting Ensemble**


## 1. Libraries

In [72]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from string import ascii_lowercase
from itertools import combinations

import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import  GradientBoostingClassifier

from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

## 2. Loading the data


In [73]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test_x.csv')

## 4. Feature Engineering

In [74]:
x_train = train.copy()
x_train.drop('voted', axis=1, inplace = True)
y_train = train['voted']

In [75]:
dataset = [x_train, test]

### 마키아밸리 테스트 FE

In [76]:
questions = [i for i in list(ascii_lowercase)[:20]]
answers = [('Q'+i+'A') for i in questions]

In [77]:
for data in dataset:
  data['T'] = data['QcA'] - data['QfA'] + data['QoA'] - data['QrA'] + data['QsA']
  data['V'] = data['QbA'] - data['QeA'] + data['QhA'] + data['QjA'] + data['QmA'] - data['QqA']
  # data['M'] = - data['QkA']

Tactic/ Morality/ View에 따라 feature 항목을 나눠보았습니다.

In [78]:
flipping_columns = ["QeA", "QfA", "QkA", "QqA", "QrA"]
for data in dataset:
  for flip in flipping_columns: 
    data[flip] = 6 - data[flip]

In [79]:
flipping_secret_columns = ["QaA", "QdA", "QgA", "QiA", "QnA"]
for data in dataset:
  for flip in flipping_secret_columns: 
    data[flip] = 6 - data[flip]

In [80]:
for data in dataset:
  data['Mach_score'] = data[answers].mean(axis = 1)

In [81]:
for data in dataset:
  data['delay'] = data[[('Q'+i+'E') for i in questions]].sum(axis=1)
  data['delay'] = data['delay'] ** (1/10)

In [82]:
Ancoms = list(combinations(answers, 2))
for data in dataset:
  for a,b in Ancoms:
    data['%s_dv_%s'%(a,b)] = data[a]/data[b]

In [83]:
for data in dataset:
  data.drop([('Q'+i+'A') for i in questions], axis = 1, inplace = True)
  data.drop([('Q'+i+'E') for i in questions], axis = 1, inplace = True)

### 나머지 Features


In [84]:
for data in dataset:
  data.drop('hand', axis=1, inplace = True)

In [85]:
wr_list = [('wr_0'+str(i)) for i in range(1,10)]
wr_list.extend([('wr_'+str(i)) for i in range(10,14)])
wr_no_need = [i for i in wr_list if i not in ['wr_01', 'wr_03', 'wr_06', 'wr_09', 'wr_11']]

EDA에서 결과에 큰 영향이 없다고 판단된 feature들을 제거해주었습니다.

In [86]:
for data in dataset:
  data.drop(wr_no_need, axis=1, inplace = True)

In [87]:
for data in dataset:
  data['Ex'] = (data['tp01']+data['tp06'])/2
  data['Ag'] = (data['tp07']+data['tp02'])/2
  data['Con'] = (data['tp03']+data['tp08'])/2
  data['Es'] =(data['tp09']+data['tp04'])/2
  data['Op'] =(data['tp05']+data['tp10'])/2

TIPI test에 따라 feature 항목을 나눠놓았는데, 이때는 tipi feature들이 flip된 형태로 저장되어있는지 몰라서 따로 전처리를 해주지 않았었습니다.

In [88]:
for data in dataset:
  data.drop([('tp0'+str(i)) for i in range(1,10)], axis=1, inplace = True)
  data.drop('tp10', axis = 1, inplace = True)

In [89]:
index = test['index']
for data in dataset:
  data.drop('index', axis = 1, inplace = True)

In [90]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
needenco = ['age_group', 'gender', 'race', 'religion']
for i in needenco:
  x_train[i] = encoder.fit_transform(x_train[i])
  test[i] = encoder.transform(test[i])

# SHAP


In [ ]:
# from catboost import CatBoostClassifier
# import shap
# import matplotlib.pyplot as plt
# plt.rc('font', family='Malgun Gothic')

# X = x_train.copy()
# y = list(map(lambda x: 0 if x==1 else 1, y_train))

# model = CatBoostClassifier(
#     # iterations=5, 
#     # learning_rate=0.1, 
#     # loss_function='CrossEntropy'
# )
# model.fit(X, y)

# # explain the model's predictions using SHAP
# # (same syntax works for LightGBM, CatBoost, scikit-learn, transformers, Spark, etc.)
# explainer = shap.TreeExplainer(model)
# shap_values = explainer(X)
# shap.initjs()

# # visualize the first prediction's explanation
# shap.plots.waterfall(shap_values[0])
# shap.summary_plot(shap_values, X, plot_type="bar")
# shap.plots.force(shap_values[0])

In [ ]:
# import numpy as np

# vals = np.abs(shap_values.values).mean(0)
# feature_names = X.columns

# feature_importance = pd.DataFrame(list(zip(feature_names, vals)),columns=['col_name','feature_importance_vals'])
# feature_importance.sort_values(by=['feature_importance_vals'],ascending=False, inplace=True)
# feature_importance.head()

In [ ]:
# shap.plots.bar(shap_values, max_display=300)

In [20]:
# shap.plots.beeswarm(shap_values, max_display=350)

In [ ]:
# feature_importance[feature_importance['col_name']=='QmA_dv_QpA']

In [ ]:
# temp_ls = []
# for i in feature_importance.index:
#     if i!=185:
#         temp_ls.append(feature_importance.loc[i]['col_name'])
#     else:
#         break

In [ ]:
# len(temp_ls)

## 5. Model

In [91]:
from pycaret.classification import *

x_train['target'] = list(map(lambda x: 0 if x==1 else 1,y_train))

In [92]:
setup_clf = setup(data=x_train, target='target')

,Description,Value
0,Session id,4454
1,Target,target
2,Target type,Binary
3,Original data shape,"(45532, 217)"
4,Transformed data shape,"(45532, 217)"
5,Transformed train set shape,"(31872, 217)"
6,Transformed test set shape,"(13660, 217)"
7,Numeric features,216
8,Preprocess,True
9,Imputation type,simple


In [93]:
best_3 = compare_models(sort = 'AUC', n_select = 3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.6939,0.7650,0.6451,0.7590,0.6974,0.3917,0.3971,4.2870
catboost,CatBoost Classifier,0.6920,0.7624,0.6581,0.7484,0.7003,0.3863,0.3897,22.2850
lightgbm,Light Gradient Boosting Machine,0.6921,0.7609,0.6422,0.7579,0.6952,0.3884,0.3939,0.4730
ada,Ada Boost Classifier,0.6883,0.7577,0.6441,0.7506,0.6932,0.3801,0.3847,0.9410
lda,Linear Discriminant Analysis,0.6713,0.7423,0.7236,0.6904,0.7066,0.3335,0.3341,0.2710
lr,Logistic Regression,0.6717,0.7418,0.7251,0.6903,0.7072,0.3342,0.3348,1.7110
rf,Random Forest Classifier,0.6732,0.7377,0.6580,0.7202,0.6877,0.3464,0.3480,0.4330
et,Extra Trees Classifier,0.6695,0.7344,0.6958,0.6986,0.6972,0.3335,0.3335,0.6070
qda,Quadratic Discriminant Analysis,0.4863,0.6202,0.1383,0.6887,0.1873,0.0419,0.0813,0.2160
nb,Naive Bayes,0.4665,0.6059,0.0672,0.0611,0.0640,0.0157,0.0158,0.1570


Processing:   0%|          | 0/67 [00:00<?, ?it/s]

In [94]:
blended = blend_models(estimator_list = best_3, fold = 5, method = 'soft')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.6899,0.7630,0.6386,0.7564,0.6925,0.3842,0.3898
1,0.7016,0.7686,0.6486,0.7696,0.7039,0.4076,0.4137
2,0.6897,0.7627,0.6509,0.7488,0.6964,0.3822,0.3861
3,0.6875,0.7614,0.6367,0.7535,0.6902,0.3794,0.3849
4,0.7010,0.7727,0.6545,0.7647,0.7053,0.4055,0.4106
Mean,0.6939,0.7657,0.6459,0.7586,0.6977,0.3918,0.3970
Std,0.0061,0.0043,0.0070,0.0075,0.0060,0.0122,0.0125


Processing:   0%|          | 0/6 [00:00<?, ?it/s]

In [95]:
pred_holdout = predict_model(blended)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.6920,0.7592,0.6482,0.7541,0.6971,0.3875,0.3921


In [96]:
final_model = finalize_model(blended)

In [97]:
predictions = predict_model(final_model, data = test)

In [98]:
predictions['Score']

0        0.7073
1        0.8880
2        0.5873
3        0.7865
4        0.7393
          ...  
11378    0.5603
11379    0.8507
11380    0.7638
11381    0.6889
11382    0.5133
Name: Score, Length: 11383, dtype: float64

In [100]:
submission = pd.DataFrame({
    "index" : index,
    "voted" : predictions['Score']
})
submission.to_csv('./data/model1.csv', index=False)

In [21]:
k_fold = KFold(n_splits = 3, shuffle = True, random_state = 0)

In [24]:
from catboost import CatBoostClassifier

clf1 = RandomForestClassifier(n_estimators=500)
clf2 = LGBMClassifier()
clf3 = GradientBoostingClassifier()
clf4 = CatBoostClassifier()

soft_vote  = VotingClassifier([('r1',clf1), ('r2', clf2), ('r3',clf3), ('r4',clf4)], voting='soft')
soft_vote.fit(x_train, y_train)

Learning rate set to 0.052606
0:	learn: 0.6769188	total: 67.5ms	remaining: 1m 7s
1:	learn: 0.6658066	total: 85.8ms	remaining: 42.8s
2:	learn: 0.6559653	total: 100ms	remaining: 33.3s
3:	learn: 0.6474527	total: 114ms	remaining: 28.4s
4:	learn: 0.6365732	total: 128ms	remaining: 25.5s
5:	learn: 0.6277312	total: 143ms	remaining: 23.7s
6:	learn: 0.6201719	total: 156ms	remaining: 22.2s
7:	learn: 0.6153279	total: 169ms	remaining: 21s
8:	learn: 0.6108046	total: 183ms	remaining: 20.1s
9:	learn: 0.6075068	total: 195ms	remaining: 19.3s
10:	learn: 0.6046391	total: 208ms	remaining: 18.7s
11:	learn: 0.6013761	total: 221ms	remaining: 18.2s
12:	learn: 0.5988660	total: 231ms	remaining: 17.6s
13:	learn: 0.5957185	total: 243ms	remaining: 17.1s
14:	learn: 0.5919632	total: 256ms	remaining: 16.8s
15:	learn: 0.5889215	total: 267ms	remaining: 16.4s
16:	learn: 0.5873996	total: 277ms	remaining: 16s
17:	learn: 0.5853914	total: 289ms	remaining: 15.7s
18:	learn: 0.5837983	total: 299ms	remaining: 15.4s
19:	learn: 0.

VotingClassifier(estimators=[('r1', RandomForestClassifier(n_estimators=500)),
                             ('r2', LGBMClassifier()),
                             ('r3', GradientBoostingClassifier()),
                             ('r4',
                              <catboost.core.CatBoostClassifier object at 0x7f3deef8dac0>)],
                 voting='soft')

In [26]:
model = soft_vote
pred_y = model.predict_proba(test)
pred_y = pred_y[:,1]

submission = pd.DataFrame({
    "index" : index,
    "voted" : pred_y
})
submission.to_csv('./data/model1.csv', index=False)


# **Model2: Lgbm Ensemble with different features**

## 1. Libraries

In [27]:
import pandas as pd
import numpy as np

from string import ascii_lowercase
from itertools import combinations

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

In [28]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
import eli5
from eli5.sklearn import PermutationImportance

import matplotlib.pyplot as plt

import warnings
import gc
warnings.filterwarnings("ignore")

## 2. Loading the data

In [29]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test_x.csv')

## 3. Feature Engineering

In [30]:
x_train = train.copy()
x_train.drop('voted', axis=1, inplace = True)
y_train = train['voted']

In [31]:
dataset = [x_train, test]





### 마키아밸리 테스트 FE

In [32]:
questions = [i for i in list(ascii_lowercase)[:20]]
answers = [('Q'+i+'A') for i in questions]

In [33]:
for data in dataset:
  data['T'] = data['QcA'] - data['QfA'] + data['QoA'] - data['QrA'] + data['QsA']
  data['V'] = data['QbA'] - data['QeA'] + data['QhA'] + data['QjA'] + data['QmA'] - data['QqA']
  # data['M'] = - data['QkA']

In [34]:
flipping_columns = ["QeA", "QfA", "QkA", "QqA", "QrA"]
for data in dataset:
  for flip in flipping_columns: 
    data[flip] = 6 - data[flip]

In [35]:
flipping_secret_columns = ["QaA", "QdA", "QgA", "QiA", "QnA"]
for data in dataset:
  for flip in flipping_secret_columns: 
    data[flip] = 6 - data[flip]

In [36]:
for data in dataset:
  data['Mach_score'] = data[answers].mean(axis = 1)

In [37]:
for data in dataset:
  data['delay'] = data[[('Q'+i+'E') for i in questions]].sum(axis=1)
  data['delay'] = data['delay'] ** (1/10)
  data['delay_var'] = data['delay'].var()

In [38]:

Ancoms = list(combinations(answers, 2))
for data in dataset:
  for a,b in Ancoms:
    data['mach_%s_dv_%s'%(a,b)] = data[a]/data[b]

In [39]:
for data in dataset:
  data['mach_var'] = data[answers].var(axis=1)


### 나머지 Features


In [40]:
tps = ['tp01', 'tp02', 'tp03', 'tp04', 'tp05', 'tp06', 'tp07', 'tp08', 'tp09', 'tp10']
for data in dataset:
  for tp in tps:
    data[tp] = 7 - data[tp]

tipi feature들을 일반적인 형태로 복구시켜줬습니다.

In [41]:
for data in dataset:
  for tp in tps:
    data[tp] = data[tp].replace(0, np.nan)
    mean = data[tp].mean(axis=0)
    data[tp] = data[tp].replace(np.nan , mean)


tp중 무응답 값들을 평균값으로 대체했습니다.

In [42]:
for data in dataset:
  data['Ex'] = (data['tp01']+data['tp06'])/2
  data['Ag'] = (data['tp07']-data['tp02'])/2
  data['Con'] = (data['tp03']-data['tp08'])/2
  data['Es'] =(data['tp09']-data['tp04'])/2
  data['Op'] =(data['tp05']-data['tp10'])/2

In [43]:
index = test['index']
for data in dataset:
  data.drop('index', axis = 1, inplace = True)

In [44]:
import numpy as np
for data in dataset:
  teenager_ox = 1*np.array(data['age_group'] == '10s')
  data['teenager_ox'] = teenager_ox

10대인지 아닌지의 여부가 투표 여부에 큰 영향을 미칠 것 같아 하나의 column을 더 만들어주었습니다. 

In [45]:
tpcoms = list(combinations(tps, 2))
for data in dataset:
  for a,b in tpcoms:
    data['tp_%s_dv_%s'%(a,b)] = data[a]/data[b]

tp 값들끼리 나눈 feature들을 생성해주었습니다.

In [46]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
needenco = ['age_group', 'gender', 'race', 'religion']
for i in needenco:
  x_train[i] = encoder.fit_transform(x_train[i])
  test[i] = encoder.transform(test[i])

In [47]:
for data in dataset:
  data['Es_gender'] = data['Es']*data['gender']
  data['Con_gender'] = data['Con']*data['gender']
  data['Op_gender'] = data['Op']*data['gender']

# SHAP

In [ ]:
from catboost import CatBoostClassifier
import shap
import matplotlib.pyplot as plt
plt.rc('font', family='Malgun Gothic')

X = x_train.copy()
y = list(map(lambda x: 0 if x==1 else 1, y_train))

model = CatBoostClassifier()
model.fit(X, y)

explainer = shap.TreeExplainer(model)
shap_values = explainer(X)
shap.initjs()

In [ ]:
import numpy as np

vals = np.abs(shap_values.values).mean(0)
feature_names = X.columns

feature_importance = pd.DataFrame(list(zip(feature_names, vals)),columns=['col_name','feature_importance_vals'])
feature_importance.sort_values(by=['feature_importance_vals'],ascending=False, inplace=True)
feature_importance.head()

In [ ]:
shap.plots.beeswarm(shap_values, max_display=350)

In [ ]:
# shap.plots.bar(shap_values, max_display=300)

In [ ]:
feature_importance[feature_importance['col_name']=='wr_08']

In [ ]:
temp_ls = []
for i in feature_importance.index:
    if i!=70:
        temp_ls.append(feature_importance.loc[i]['col_name'])
    else:
        break

In [ ]:
len(temp_ls)

EDA 결과, 성별에 따라 Emotional Stability/ Conscience/ Open Minded가 투표 여부에 미치는 영향이 크다고 판단되어 feature를 추가해주었습니다.

정보 출처: https://www.sciencedirect.com/science/article/abs/pii/S0261379413001613

## 4. Feature Selection 1 & Model 2-1

In [48]:
x_train = x_train
test = test

In [49]:
def lgbm_rfe_4040(x_data, y_data, ratio=0.9, min_feats=40):
    feats = x_data.columns.tolist()
    archive = pd.DataFrame(columns=['model', 'n_feats', 'feats', 'score'])
    while True:
        model = LGBMClassifier(objective = 'binary', num_iterations=10**4)
        x_train, x_val, y_train, y_val = train_test_split(x_data[feats], y_data, random_state=4040)
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], early_stopping_rounds=100, verbose=False)
        val_pred = model.predict_proba(x_val)
        val_pred = val_pred[:,1]
        score = roc_auc_score(y_val, val_pred)
        n_feats = len(feats)
        print(n_feats, score)
        archive = archive.append({'model': model, 'n_feats': n_feats, 'feats': feats, 'score': score}, ignore_index=True)
        feat_imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)        
        next_n_feats = int(n_feats * ratio)
        if next_n_feats < min_feats:
            break
        else:
            feats = feat_imp.iloc[:next_n_feats].index.tolist()
    return archive


In [50]:
lgbm_archive_4040 = lgbm_rfe_4040(x_train, y_train)

326 0.7657604433861234
293 0.7657604433861234
263 0.7660072793456507
236 0.767402121602152
212 0.7663034016997157
190 0.7662282912232989
171 0.7669648213852982
153 0.7663662371913642
137 0.764458673482484
123 0.7650298021092214
110 0.7662878015924913
99 0.7658491651114727
89 0.7650935854161497
80 0.764219451121104
72 0.7654788645583153
64 0.7658633046508891
57 0.7674912628522975
51 0.7673306314693442
45 0.7654532580078116
40 0.7655709269440764


In [51]:
lgbm_archive_4040

,model,n_feats,feats,score
0,"LGBMClassifier(num_iterations=10000, objective...",326,"[QaA, QaE, QbA, QbE, QcA, QcE, QdA, QdE, QeA, ...",0.765760
1,"LGBMClassifier(num_iterations=10000, objective...",293,"[education, race, religion, teenager_ox, marri...",0.765760
2,"LGBMClassifier(num_iterations=10000, objective...",263,"[education, race, religion, teenager_ox, marri...",0.766007
3,"LGBMClassifier(num_iterations=10000, objective...",236,"[education, race, religion, teenager_ox, marri...",0.767402
4,"LGBMClassifier(num_iterations=10000, objective...",212,"[education, race, religion, married, teenager_...",0.766303
5,"LGBMClassifier(num_iterations=10000, objective...",190,"[education, race, religion, teenager_ox, marri...",0.766228
6,"LGBMClassifier(num_iterations=10000, objective...",171,"[education, race, religion, teenager_ox, marri...",0.766965
7,"LGBMClassifier(num_iterations=10000, objective...",153,"[education, race, religion, teenager_ox, marri...",0.766366
8,"LGBMClassifier(num_iterations=10000, objective...",137,"[education, race, religion, married, teenager_...",0.764459
9,"LGBMClassifier(num_iterations=10000, objective...",123,"[education, race, religion, QlE, QhE, QkE, tee...",0.765030


In [52]:
model = LGBMClassifier(objective="binary", num_iterations= 10**3)

x_train_1 = x_train[lgbm_archive_4040.iloc[16,2]]

model.fit(x_train_1, y_train)

pred_y1 = model.predict_proba(test[lgbm_archive_4040.iloc[16,2]])
pred_y1 = pred_y1[:,1]

## 5. Feature Selection 2 & Model 2-2

In [53]:
def lgbm_rfe_1234(x_data, y_data, ratio=0.9, min_feats=40):
    feats = x_data.columns.tolist()
    archive = pd.DataFrame(columns=['model', 'n_feats', 'feats', 'score'])
    while True:
        model = LGBMClassifier(objective = 'binary', num_iterations=10**4)
        x_train, x_val, y_train, y_val = train_test_split(x_data[feats], y_data, random_state=1234)
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], early_stopping_rounds=100, verbose=False)
        val_pred = model.predict_proba(x_val)
        val_pred = val_pred[:,1]
        score = roc_auc_score(y_val, val_pred)
        n_feats = len(feats)
        print(n_feats, score)
        archive = archive.append({'model': model, 'n_feats': n_feats, 'feats': feats, 'score': score}, ignore_index=True)
        feat_imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
        next_n_feats = int(n_feats * ratio)
        if next_n_feats < min_feats:
            break
        else:
            feats = feat_imp.iloc[:next_n_feats].index.tolist()
    return archive


In [54]:
lgbm_archive_1234 = lgbm_rfe_1234(x_train, y_train)

326 0.7602347615692592
293 0.7602347615692592
263 0.7615889599674073
236 0.7618358036388962
212 0.7616058418942144
190 0.760676619528091
171 0.760091483223663
153 0.7607334948017994
137 0.7613394220400719
123 0.7603031769792814
110 0.761862341280298
99 0.7621147136270788
89 0.7614385021307981
80 0.758855645197964
72 0.7607477447676929
64 0.7599520983113712
57 0.7593434300960527
51 0.7586410080067678
45 0.7598433469323171
40 0.7585046132512536


In [55]:
lgbm_archive_1234

,model,n_feats,feats,score
0,"LGBMClassifier(num_iterations=10000, objective...",326,"[QaA, QaE, QbA, QbE, QcA, QcE, QdA, QdE, QeA, ...",0.760235
1,"LGBMClassifier(num_iterations=10000, objective...",293,"[education, race, religion, teenager_ox, marri...",0.760235
2,"LGBMClassifier(num_iterations=10000, objective...",263,"[education, race, religion, teenager_ox, marri...",0.761589
3,"LGBMClassifier(num_iterations=10000, objective...",236,"[education, race, religion, teenager_ox, marri...",0.761836
4,"LGBMClassifier(num_iterations=10000, objective...",212,"[education, race, religion, teenager_ox, marri...",0.761606
5,"LGBMClassifier(num_iterations=10000, objective...",190,"[education, race, religion, teenager_ox, marri...",0.760677
6,"LGBMClassifier(num_iterations=10000, objective...",171,"[education, race, religion, teenager_ox, marri...",0.760091
7,"LGBMClassifier(num_iterations=10000, objective...",153,"[education, race, religion, teenager_ox, marri...",0.760733
8,"LGBMClassifier(num_iterations=10000, objective...",137,"[education, race, religion, teenager_ox, marri...",0.761339
9,"LGBMClassifier(num_iterations=10000, objective...",123,"[education, race, religion, teenager_ox, marri...",0.760303


In [56]:
model2 = LGBMClassifier(objective="binary", num_iterations= 10**3)

x_train_2 = x_train[lgbm_archive_1234.iloc[11,2]]

model2.fit(x_train_2, y_train)

pred_y2 = model2.predict_proba(test[lgbm_archive_1234.iloc[11,2]])
pred_y2 = pred_y2[:,1]

## 6. Feature Selection 3 & Model 2-3

In [57]:
def lgbm_rfe_99087(x_data, y_data, ratio=0.9, min_feats=40):
    feats = x_data.columns.tolist()
    archive = pd.DataFrame(columns=['model', 'n_feats', 'feats', 'score'])
    while True:
        model = LGBMClassifier(objective = 'binary', num_iterations=10**4)
        x_train, x_val, y_train, y_val = train_test_split(x_data[feats], y_data, random_state=99087)
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], early_stopping_rounds=100, verbose=False)
        val_pred = model.predict_proba(x_val)
        val_pred = val_pred[:,1]
        score = roc_auc_score(y_val, val_pred)
        n_feats = len(feats)
        print(n_feats, score)
        archive = archive.append({'model': model, 'n_feats': n_feats, 'feats': feats, 'score': score}, ignore_index=True)
        feat_imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
        next_n_feats = int(n_feats * ratio)
        if next_n_feats < min_feats:
            break
        else:
            feats = feat_imp.iloc[:next_n_feats].index.tolist()
    return archive


In [58]:
lgbm_archive_99087 = lgbm_rfe_99087(x_train, y_train)

326 0.7603119723843995
293 0.7605067064732967
263 0.7609278339218781
236 0.7599520310682977
212 0.7604709847297095
190 0.7603586207789662
171 0.7598230592306667
153 0.7582222581670646
137 0.7591100252116129
123 0.7598585163687459
110 0.7595193854976009
99 0.7589942836493844
89 0.7595497995659971
80 0.7601810237879729
72 0.7598538779898399
64 0.7601956393510353
57 0.7587437489283477
51 0.7581888867429902
45 0.7589496742737326
40 0.7578052554514025


In [59]:
lgbm_archive_99087

,model,n_feats,feats,score
0,"LGBMClassifier(num_iterations=10000, objective...",326,"[QaA, QaE, QbA, QbE, QcA, QcE, QdA, QdE, QeA, ...",0.760312
1,"LGBMClassifier(num_iterations=10000, objective...",293,"[education, race, teenager_ox, religion, marri...",0.760507
2,"LGBMClassifier(num_iterations=10000, objective...",263,"[education, race, teenager_ox, religion, marri...",0.760928
3,"LGBMClassifier(num_iterations=10000, objective...",236,"[education, race, religion, teenager_ox, age_g...",0.759952
4,"LGBMClassifier(num_iterations=10000, objective...",212,"[education, race, religion, teenager_ox, marri...",0.760471
5,"LGBMClassifier(num_iterations=10000, objective...",190,"[education, race, teenager_ox, religion, marri...",0.760359
6,"LGBMClassifier(num_iterations=10000, objective...",171,"[education, race, religion, teenager_ox, age_g...",0.759823
7,"LGBMClassifier(num_iterations=10000, objective...",153,"[education, race, religion, teenager_ox, marri...",0.758222
8,"LGBMClassifier(num_iterations=10000, objective...",137,"[education, race, religion, teenager_ox, age_g...",0.759110
9,"LGBMClassifier(num_iterations=10000, objective...",123,"[education, race, religion, teenager_ox, marri...",0.759859


In [60]:
model3 = LGBMClassifier(objective="binary", num_iterations= 10**3)

x_train_3 = x_train[lgbm_archive_99087.iloc[2,2]]

model3.fit(x_train_3, y_train)

pred_y3 = model3.predict_proba(test[lgbm_archive_99087.iloc[2,2]])
pred_y3 = pred_y3[:,1]

## 7. Feature Selection 4 & Model 2-4 

In [61]:
def lgbm_rfe_42(x_data, y_data, ratio=0.9, min_feats=40):
    feats = x_data.columns.tolist()
    archive = pd.DataFrame(columns=['model', 'n_feats', 'feats', 'score'])
    while True:
        model = LGBMClassifier(objective = 'binary', num_iterations=10**4)
        x_train, x_val, y_train, y_val = train_test_split(x_data[feats], y_data, random_state=42)
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], early_stopping_rounds=100, verbose=False)
        val_pred = model.predict_proba(x_val)
        val_pred = val_pred[:,1]
        score = roc_auc_score(y_val, val_pred)
        n_feats = len(feats)
        print(n_feats, score)
        archive = archive.append({'model': model, 'n_feats': n_feats, 'feats': feats, 'score': score}, ignore_index=True)
        feat_imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
        next_n_feats = int(n_feats * ratio)
        if next_n_feats < min_feats:
            break
        else:
            feats = feat_imp.iloc[:next_n_feats].index.tolist()
    return archive


In [62]:
lgbm_archive_42 = lgbm_rfe_42(x_train, y_train)

326 0.768031830395951
293 0.768031830395951
263 0.768487993930189
236 0.7691866373503328
212 0.7698177263536815
190 0.7680693744635338
171 0.768743004070823
153 0.7683404638006077
137 0.7680315813474099
123 0.7694577266874066
110 0.7673551966406341
99 0.7668502507233925
89 0.7671422134413242
80 0.7676553312638229
72 0.7669653734115528
64 0.7682408755152037
57 0.7696710056318592
51 0.7704799152936102
45 0.770109953685688
40 0.7698090407858079


In [63]:
lgbm_archive_42

,model,n_feats,feats,score
0,"LGBMClassifier(num_iterations=10000, objective...",326,"[QaA, QaE, QbA, QbE, QcA, QcE, QdA, QdE, QeA, ...",0.768032
1,"LGBMClassifier(num_iterations=10000, objective...",293,"[education, race, religion, teenager_ox, age_g...",0.768032
2,"LGBMClassifier(num_iterations=10000, objective...",263,"[education, race, religion, teenager_ox, age_g...",0.768488
3,"LGBMClassifier(num_iterations=10000, objective...",236,"[education, race, religion, teenager_ox, age_g...",0.769187
4,"LGBMClassifier(num_iterations=10000, objective...",212,"[education, race, religion, teenager_ox, age_g...",0.769818
5,"LGBMClassifier(num_iterations=10000, objective...",190,"[education, race, religion, teenager_ox, QkE, ...",0.768069
6,"LGBMClassifier(num_iterations=10000, objective...",171,"[education, race, religion, teenager_ox, age_g...",0.768743
7,"LGBMClassifier(num_iterations=10000, objective...",153,"[education, race, religion, teenager_ox, marri...",0.768340
8,"LGBMClassifier(num_iterations=10000, objective...",137,"[education, race, religion, teenager_ox, QfE, ...",0.768032
9,"LGBMClassifier(num_iterations=10000, objective...",123,"[education, race, religion, teenager_ox, age_g...",0.769458


In [64]:
model4 = LGBMClassifier(objective="binary", num_iterations= 10**3)

x_train_4 = x_train[lgbm_archive_42.iloc[17,2]]

model4.fit(x_train_4, y_train)

pred_y4 = model4.predict_proba(test[lgbm_archive_42.iloc[17,2]])
pred_y4 = pred_y4[:,1]

## 8. Ensemble

In [65]:
pred_all = (pred_y + pred_y2 + pred_y3 + pred_y4) * (1/4)

submission = pd.DataFrame({
    "index" : index,
    "voted" : pred_all
})
submission.to_csv('./data/model2.csv', index=False)

# **Model3: NN**

3번째 모델은 Junho Sun 님께서 공유해주신 코드를 그대로 활용하였습니다.

좋은 모델을 공유해주신 덕분에 public score도 0.78대로 올라갈 수 있었습니다. 
정말 감사합니다!

In [ ]:
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from tqdm import tqdm

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

drop_list = ['QaE', 'QbE', 'QcE', 'QdE', 'QeE',
             'QfE', 'QgE', 'QhE', 'QiE', 'QjE',
             'QkE', 'QlE', 'QmE', 'QnE', 'QoE',
             'QpE', 'QqE', 'QrE', 'QsE', 'QtE',
             'index', 'hand']
replace_dict = {'education': str, 'engnat': str, 'married': str, 'urban': str}
train_data = pd.read_csv('./data/train.csv')
test_data = pd.read_csv('./data/test_x.csv')
train_data = train_data.drop(train_data[train_data.familysize > 50].index)
train_y = train_data['voted']
train_x = train_data.drop(drop_list + ['voted'], axis=1)
test_x = test_data.drop(drop_list, axis=1)
train_x = train_x.astype(replace_dict)
test_x = test_x.astype(replace_dict)
train_x = pd.get_dummies(train_x)
test_x = pd.get_dummies(test_x)
train_y = 2 - train_y.to_numpy()
train_x = train_x.to_numpy()
test_x = test_x.to_numpy()

train_y_t = torch.tensor(train_y, dtype=torch.float32)
train_x_t = torch.tensor(train_x, dtype=torch.float32)
test_x_t = torch.tensor(test_x, dtype=torch.float32)
train_x_t[:, :20] = (train_x_t[:, :20] - 3.) / 2.
test_x_t[:, :20] = (test_x_t[:, :20] - 3.) / 2
train_x_t[:, 20] = (train_x_t[:, 20] - 5.) / 4.
test_x_t[:, 20] = (test_x_t[:, 20] - 5.) / 4.
train_x_t[:, 21:31] = (train_x_t[:, 21:31] - 3.5) / 3.5
test_x_t[:, 21:31] = (test_x_t[:, 21:31] - 3.5) / 3.5
test_len = len(test_x_t)

N_REPEAT = 5
N_SKFOLD = 7
N_EPOCH = 48
BATCH_SIZE = 7200
LOADER_PARAM = {
    'batch_size': BATCH_SIZE,
    'num_workers': 4,
    'pin_memory': True
}
prediction = np.zeros((test_len, 1), dtype=np.float32)

for repeat in range(N_REPEAT):

    skf, tot = StratifiedKFold(n_splits=N_SKFOLD, random_state=repeat, shuffle=True), 0.
    for skfold, (train_idx, valid_idx) in enumerate(skf.split(train_x, train_y)):
        train_idx, valid_idx = list(train_idx), list(valid_idx)
        train_loader = DataLoader(TensorDataset(train_x_t[train_idx, :], train_y_t[train_idx]),
                                  shuffle=True, drop_last=True, **LOADER_PARAM)
        valid_loader = DataLoader(TensorDataset(train_x_t[valid_idx, :], train_y_t[valid_idx]),
                                  shuffle=False, drop_last=False, **LOADER_PARAM)
        test_loader = DataLoader(TensorDataset(test_x_t, torch.zeros((test_len,), dtype=torch.float32)),
                                 shuffle=False, drop_last=False, **LOADER_PARAM)
        model = nn.Sequential(
            nn.Dropout(0.05),
            nn.Linear(91, 180, bias=False),
            nn.LeakyReLU(0.05, inplace=True),
            nn.Dropout(0.5),
            nn.Linear(180, 32, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(32, 1)
        ).to(DEVICE)
        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.20665], device=DEVICE))
        optimizer = optim.AdamW(model.parameters(), lr=5e-3, weight_decay=7.8e-2)
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=N_EPOCH // 6, eta_min=4e-4)
        prediction_t, loss_t = np.zeros((test_len, 1), dtype=np.float32), 1.

        # for epoch in range(N_EPOCH):
        for epoch in tqdm(range(N_EPOCH), desc='{:02d}/{:02d}'.format(skfold + 1, N_SKFOLD)):
            model.train()
            for idx, (xx, yy) in enumerate(train_loader):
                optimizer.zero_grad()
                xx, yy = xx.to(DEVICE), yy.to(DEVICE)
                pred = model(xx).squeeze()
                loss = criterion(pred, yy)
                loss.backward()
                optimizer.step()
                scheduler.step(epoch + idx / len(train_loader))

            with torch.no_grad():
                model.eval()
                running_acc, running_loss, running_count = 0, 0., 0
                for xx, yy in valid_loader:
                    xx, yy = xx.to(DEVICE), yy.to(DEVICE)
                    pred = model(xx).squeeze()
                    loss = criterion(pred, yy)
                    running_loss += loss.item() * len(yy)
                    running_count += len(yy)
                    running_acc += ((torch.sigmoid(pred) > 0.5).float() == yy).sum().item()
                # print('R{:02d} S{:02d} E{:02d} | {:6.4f}, {:5.2f}%'
                #       .format(repeat + 1, skfold + 1, epoch + 1, running_loss / running_count,
                #               running_acc / running_count * 100))

                if running_loss / running_count < loss_t:
                    loss_t = running_loss / running_count
                    for idx, (xx, _) in enumerate(test_loader):
                        xx = xx.to(DEVICE)
                        pred = (2. - torch.sigmoid(model(xx).detach().to('cpu'))).numpy()
                        prediction_t[BATCH_SIZE * idx:min(BATCH_SIZE * (idx + 1), len(prediction)), :] \
                            = pred[:, :].copy()
        prediction[:, :] += prediction_t[:, :].copy() / (N_REPEAT * N_SKFOLD)
        tot += loss_t
    print('R{} -> {:6.4f}'.format(repeat + 1, tot / N_SKFOLD))

df = pd.read_csv('./data/sample_submission.csv')
df.iloc[:, 1:] = prediction

In [ ]:
df.to_csv('./data/model3.csv', index=False)

# Final Ensemble

In [101]:
model1 = pd.read_csv('./data/model1.csv', index_col = 'index')
model2 = pd.read_csv('./data/model2.csv', index_col='index')

pred_y = (model1)*(0.7) + (model2)*(0.3)

test = pd.read_csv('./data/test_x.csv')
index = test['index']

submission = pd.DataFrame({
    'index': index,
    'voted': pred_y['voted']
    })

submission.to_csv('./data/combined_model1_model2.csv', index=False)

In [67]:

combined_12 = pd.read_csv('./data/combined_model1_model2.csv', index_col = 'index')
model3 = pd.read_csv('./data/model3.csv', index_col='index')
model3['voted'] = model3['voted']-1

다른 모델과 같이 [0,1]의 범위(voted가 2일 확률)를 맞춰주기 위해 1을 빼주었습니다.

In [68]:
pred_y = (model3)*(0.8) + (model1)*(0.2)

test = pd.read_csv('./data/test_x.csv')
index = test['index']

submission = pd.DataFrame({
    'index': index,
    'voted': pred_y['voted']
    })

submission.to_csv('./data/submission_final_FE.csv', index=False)